In [19]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

import datetime
# Create a logs directory
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

In [20]:
# ── 1. CONFIGURATION ──
CSV_PATH = '/Users/shubhbhateja/College/minor_project/NeoJaundice/chd_jaundice_published_2.csv'
IMAGE_DIR = '/Users/shubhbhateja/College/minor_project/images'


In [21]:
INPUT_SIZE = (128, 128)
SEQ_LEN = 128 * 128
# New: Metadata columns from your CSV
CLINICAL_COLS = ['gestational_age', 'age(day)', 'weight']

In [22]:
# Load the dataset
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows from CSV")

Loaded 2235 rows from CSV


In [23]:
def white_balance(img):
    """Corrects lighting using the Gray World algorithm."""
    result = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    avg_a = np.average(result[:, :, 1])
    avg_b = np.average(result[:, :, 2])
    result[:, :, 1] = result[:, :, 1] - ((avg_a - 128) * (result[:, :, 0] / 255.0) * 1.1)
    result[:, :, 2] = result[:, :, 2] - ((avg_b - 128) * (result[:, :, 0] / 255.0) * 1.1)
    return cv2.cvtColor(result, cv2.COLOR_LAB2BGR)

In [24]:
def adaptive_crop_skin(image_rgb):
    """Isolates the clean skin patch, ignoring card patches and dark borders."""
    h, w = image_rgb.shape[:2]
    img_ycrcb = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2YCrCb)
    
    # Skin color detection mask
    lower_skin = np.array([0, 135, 85], dtype=np.uint8)
    upper_skin = np.array([255, 175, 135], dtype=np.uint8)
    mask = cv2.inRange(img_ycrcb, lower_skin, upper_skin)
    
    # Pre-emptive masking to remove known non-skin areas
    mask[int(h*0.80):h, :] = 0  # Cut bottom 20% (calibration card)
    mask[0:int(h*0.05), :] = 0  # Cut top 5% border
    mask[:, 0:int(w*0.05)] = 0  # Cut left 5% border
    mask[:, int(w*0.95):w] = 0  # Cut right 5% border

    # Morphological cleaning & shrinking
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.erode(mask, kernel, iterations=3) # Pulls away from background edges
    
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        # Fallback if detection fails: take center crop
        return image_rgb[int(h*0.2):int(h*0.7), int(w*0.2):int(w*0.8)]
        
    largest_cnt = max(contours, key=cv2.contourArea)
    x, y, w_box, h_box = cv2.boundingRect(largest_cnt)
    
    # Apply 10% inner inset for a 'pure' skin sample
    ix, iy = int(w_box * 0.1), int(h_box * 0.1)
    return image_rgb[y+iy : y+h_box-iy, x+ix : x+w_box-ix]

In [25]:
def prepare_image_multimodal(img_path):
    img = cv2.imread(img_path)
    if img is None: return None
    
    # 1. Calibrate & Crop
    balanced = white_balance(img)
    rgb_img = cv2.cvtColor(balanced, cv2.COLOR_BGR2RGB)
    skin_patch = adaptive_crop_skin(rgb_img)
    
    # 2. Resize
    resized_rgb = cv2.resize(skin_patch, (128, 128))
    resized_hsv = cv2.cvtColor(resized_rgb, cv2.COLOR_RGB2HSV).astype('float32')
    
    # 3. Correct HSV Scaling (Bug 3 Fix)
    hsv_normalized = np.zeros_like(resized_hsv)
    hsv_normalized[:,:,0] = resized_hsv[:,:,0] / 179.0  # Hue is 0-179
    hsv_normalized[:,:,1] = resized_hsv[:,:,1] / 255.0  # Saturation 0-255
    hsv_normalized[:,:,2] = resized_hsv[:,:,2] / 255.0  # Value 0-255
    
    rgb_normalized = resized_rgb.astype('float32') / 255.0
    
    # Flatten to 1D sequence (16384, 6)
    combined_seq = np.concatenate([
        rgb_normalized.reshape(-1, 3), 
        hsv_normalized.reshape(-1, 3)
    ], axis=-1)
    
    return combined_seq

In [26]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

X_img, X_clin, Y_raw = [], [], []

# Bug 4 Fix: Ensure all arrays match index by only appending if image exists
for idx, row in df.iterrows():
    path = os.path.join(IMAGE_DIR, row['image_idx'])
    img_data = prepare_image_multimodal(path)
    
    if img_data is not None:
        X_img.append(img_data)
        X_clin.append([row['gestational_age'], row['age(day)'], row['weight']])
        Y_raw.append(row['blood(mg/dL)'])

X_img = np.array(X_img)
X_clin = np.array(X_clin)
Y_raw = np.array(Y_raw).reshape(-1, 1)

# --- 70/10/20 Split (Bug 2 Fix) ---
# First split: 80% (Train+Val) and 20% (Test)
X_i_temp, X_i_test, X_c_temp, X_c_test, y_temp, y_test = train_test_split(
    X_img, X_clin, Y_raw, test_size=0.2, random_state=42
)

# Second split: From the 80%, take 12.5% for Val (results in 10% of total)
X_i_train, X_i_val, X_c_train, X_c_val, y_train, y_val = train_test_split(
    X_i_temp, X_c_temp, y_temp, test_size=0.125, random_state=42
)

# --- Bug 1 Fix: Fit scalers ONLY on training data ---
scaler_x = StandardScaler().fit(X_c_train)
scaler_y = MinMaxScaler().fit(y_train)

X_c_train = scaler_x.transform(X_c_train)
X_c_val = scaler_x.transform(X_c_val)
X_c_test = scaler_x.transform(X_c_test)

y_train_scaled = scaler_y.transform(y_train).flatten()
y_val_scaled = scaler_y.transform(y_val).flatten()
y_test_scaled = scaler_y.transform(y_test).flatten()

In [ ]:
# def build_multimodal_1dcnn():
#     img_input = layers.Input(shape=(16384, 6), name="skin_sequence_input")
#     x = layers.Conv1D(32, 3, padding='same', activation='relu')(img_input)
#     x = layers.MaxPooling1D(2, padding='same')(x)
#     x = layers.Conv1D(64, 3, padding='same', activation='relu')(x)
#     x = layers.MaxPooling1D(2, padding='same')(x)
#     x = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
#     x = layers.MaxPooling1D(2, padding='same')(x)
#     x = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
#     x = layers.MaxPooling1D(2, padding='same')(x)
    
#     vis_feat = layers.Flatten()(x)
#     vis_feat = layers.Dense(256, activation='relu')(vis_feat)
#     vis_feat = layers.Dropout(0.3)(vis_feat)

#     clin_input = layers.Input(shape=(3,), name="clinical_input")
#     c = layers.Dense(32, activation='relu')(clin_input)
#     clin_feat = layers.Dense(16, activation='relu')(c)

#     merged = layers.Concatenate()([vis_feat, clin_feat])
#     head = layers.Dense(64, activation='relu')(merged)
#     output = layers.Dense(1, activation='linear', name="tsb_prediction")(head)

#     model = models.Model(inputs=[img_input, clin_input], outputs=output)
    
#     # CRITICAL FIX: Lower learning rate and add clipnorm
#     opt = tf.keras.optimizers.Adam(learning_rate=0.0001, clipnorm=1.0)
    
#     model.compile(optimizer=opt, loss='mse', metrics=['mae'])
#     return model

# model = build_multimodal_1dcnn()

def build_multimodal_1dcnn():
    # --- Visual Branch (Your Original 4-Layer Architecture) ---
    img_input = layers.Input(shape=(16384, 6), name="skin_sequence_input")
    
    # Layer 1: 32 filters
    x = layers.Conv1D(32, 3, padding='same')(img_input)
    x = layers.BatchNormalization()(x) # Keeps math stable
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(2, padding='same')(x)
    
    # Layer 2: 64 filters
    x = layers.Conv1D(64, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(2, padding='same')(x)
    
    # Layer 3: 128 filters
    x = layers.Conv1D(128, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(2, padding='same')(x)
    
    # Layer 4: 128 filters
    x = layers.Conv1D(128, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(2, padding='same')(x)
    
    vis_feat = layers.Flatten()(x)
    vis_feat = layers.Dense(256, activation='relu')(vis_feat)
    vis_feat = layers.Dropout(0.3)(vis_feat)

    # --- Clinical Branch ---
    clin_input = layers.Input(shape=(3,), name="clinical_input")
    c = layers.Dense(32, activation='relu')(clin_input)
    clin_feat = layers.Dense(16, activation='relu')(c)

    # --- Fusion & Output ---
    merged = layers.Concatenate()([vis_feat, clin_feat])
    head = layers.Dense(64, activation='relu')(merged)
    output = layers.Dense(1, activation='linear', name="tsb_prediction")(head)

    model = models.Model(inputs=[img_input, clin_input], outputs=output)
    
    # Using a slightly lower learning rate to prevent the "NaN" explosion
    opt = tf.keras.optimizers.Adam(learning_rate=0.0001, clipnorm=1.0)
    
    model.compile(optimizer=opt, loss='mse', metrics=['mae'])
    return model

model = build_multimodal_1dcnn()
model.summary()

In [16]:
# The model is already built in the previous cell
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ skin_sequence_input │ (None, 16384, 6)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 16384, 32) │        608 │ skin_sequence_in… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 16384, 32) │        128 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 16384, 32) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 4096, 32)  │          0 │ activation[0][0]  │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 4096, 64)  │      6,208 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 4096, 64)  │        256 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 4096, 64)  │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 1024, 64)  │          0 │ activation_1[0][… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 1024, 128) │     24,704 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1024, 128) │        512 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 1024, 128) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ activation_2[0][… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ clinical_input      │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      8,256 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │         64 │ clinical_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 8)         │        136 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 72)        │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dense_2[0][0]   

 Total params: 43,241 (168.91 KB)

 Trainable params: 42,793 (167.16 KB)

 Non-trainable params: 448 (1.75 KB)

In [17]:
# ── 4. NUMERICAL SAFETY CHECK ──
# Using the updated 70/10/20 split variable names

print(f"NaNs in Training Images: {np.isnan(X_i_train).any()}")
print(f"NaNs in Clinical Data: {np.isnan(X_c_train).any()}")
print(f"NaNs in Training Labels: {np.isnan(y_train_scaled).any()}")

# Check Validation set as well since Bug 1 caused NaNs there previously
print(f"NaNs in Validation Labels: {np.isnan(y_val_scaled).any()}")

# Filling missing values if they exist (using training mean to avoid leakage)
if np.isnan(X_c_train).any():
    print("Fixing missing clinical values in training set...")
    col_mean = np.nanmean(X_c_train, axis=0)
    inds = np.where(np.isnan(X_c_train))
    X_c_train[inds] = np.take(col_mean, inds[1])
    
if np.isnan(X_c_val).any():
    print("Fixing missing clinical values in validation set...")
    # Use training mean here too to maintain strict isolation
    col_mean = np.nanmean(X_c_train, axis=0) 
    inds = np.where(np.isnan(X_c_val))
    X_c_val[inds] = np.take(col_mean, inds[1])

NaNs in Training Images: False
NaNs in Clinical Data: True
NaNs in Training Labels: False
NaNs in Validation Labels: False
Fixing missing clinical values in training set...


In [ ]:
# ── 5. STABILIZED MULTIMODAL TRAINING WITH LOGGING ──

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, min_lr=1e-7),
    tf.keras.callbacks.TerminateOnNaN(),
    tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
    ]

history = model.fit(
    x={'skin_sequence_input': X_i_train, 'clinical_input': X_c_train},
    y=y_train_scaled,
    validation_data=(
        {'skin_sequence_input': X_i_val, 'clinical_input': X_c_val}, 
        y_val_scaled
    ),
    epochs=100,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 30s 288ms/step - loss: 8.4496 - mae: 1.4250 - val_loss: 0.0804 - val_mae: 0.2445 - learning_rate: 1.0000e-04
Epoch 2/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 28s 286ms/step - loss: 0.1064 - mae: 0.2326 - val_loss: 0.0872 - val_mae: 0.2555 - learning_rate: 1.0000e-04
Epoch 3/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 27s 279ms/step - loss: 0.0605 - mae: 0.1897 - val_loss: 0.0701 - val_mae: 0.2265 - learning_rate: 1.0000e-04
Epoch 4/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 27s 275ms/step - loss: 0.0510 - mae: 0.1771 - val_loss: 0.0492 - val_mae: 0.1852 - learning_rate: 1.0000e-04
Epoch 5/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 28s 282ms/step - loss: 0.0435 - mae: 0.1647 - val_loss: 0.0404 - val_mae: 0.1649 - learning_rate: 1.0000e-04
Epoch 6/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 27s 270ms/step - loss: 0.0416 - mae: 0.1611 - val_loss: 0.0419 - val_mae: 0.1673 - learning_rate: 1.0000e-04
Epoch 7/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 28s 281ms/step - loss: 0.0385 - mae: 0.1563 - val_loss: 0.0330 - val_mae:

In [ ]:
# ── 6. EVALUATION ──
print("\nEvaluating on Test Set...")
results = model.evaluate(
    x={'skin_sequence_input': X_i_test, 'clinical_input': X_c_test},
    y=y_test_scaled,
    verbose=1
)
print(f"Test MSE (Scaled): {results[0]:.4f}")
print(f"Test MAE (Scaled): {results[1]:.4f}")


Evaluating on Test Set...
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 60ms/step - loss: nan - mae: nan
Test MSE (Scaled): nan
Test MAE (Scaled): nan


In [ ]:
# Convert back to real mg/dL to see if we hit the 96% target
preds_scaled = model.predict({'skin_sequence_input': X_i_test, 'clinical_input': X_c_test})
preds_real = scaler_y.inverse_transform(preds_scaled)
y_real = scaler_y.inverse_transform(y_test_scaled.reshape(-1, 1))

mae_real = np.mean(np.abs(preds_real - y_real))
print(f"Final Prediction Error: {mae_real:.2f} mg/dL")

14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step
Final Prediction Error: nan mg/dL


In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/fit

ERROR: Failed to launch TensorBoard (exited with 1).
Contents of stderr:
Traceback (most recent call last):
  File "/Users/shubhbhateja/College/minor_project/.venv/bin/tensorboard", line 5, in <module>
    from tensorboard.main import run_main
  File "/Users/shubhbhateja/College/minor_project/.venv/lib/python3.13/site-packages/tensorboard/main.py", line 27, in <module>
    from tensorboard import default
  File "/Users/shubhbhateja/College/minor_project/.venv/lib/python3.13/site-packages/tensorboard/default.py", line 30, in <module>
    import pkg_resources
ModuleNotFoundError: No module named 'pkg_resources'

In [ ]:
# ── 7. PREDICTION & INVERSE SCALING ──

# Generate predictions on the test set
preds_scaled = model.predict({'skin_sequence_input': X_i_test, 'clinical_input': X_c_test})

# Convert back to actual mg/dL using the fitted scaler
preds_real = scaler_y.inverse_transform(preds_scaled)
y_real = scaler_y.inverse_transform(y_test_scaled.reshape(-1, 1))

# Final Error Calculation in clinical units (mg/dL)
mae_mgdl = np.mean(np.abs(preds_real - y_real))

print(f"Final Prediction Error: {mae_mgdl:.2f} mg/dL")

# Display first 10 samples for manual verification
comparison = pd.DataFrame({
    'Actual (mg/dL)': y_real.flatten()[:10],
    'Predicted (mg/dL)': preds_real.flatten()[:10]
})
print("\nSample Comparisons:")
print(comparison)

14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 82ms/step
Final Prediction Error: nan mg/dL

Sample Comparisons:
   Actual (mg/dL)  Predicted (mg/dL)
0             2.5           2.705346
1            17.8          12.146945
2            18.2          13.533953
3            19.4          10.508675
4             1.7           3.099476
5            10.6          13.984525
6            10.0           8.781280
7             2.4           9.126055
8             8.3          10.833567
9            10.5           8.538601


<h1>**IGNORE**</h1>

In [ ]:
# Predict on test set
preds_scaled = model.predict({'skin_sequence_input': X_i_ts, 'clinical_input': X_c_ts})

# Convert back to actual mg/dL
preds_real = scaler_y.inverse_transform(preds_scaled)
y_real = scaler_y.inverse_transform(y_ts.reshape(-1, 1))

# Final Error Calculation
mae = np.mean(np.abs(preds_real - y_real))
print(f"Final Prediction Error: {mae:.2f} mg/dL")

NameError: name 'X_i_ts' is not defined

In [ ]:
# ── VERIFICATION: WHAT DOES THE AI SEE? ──
import matplotlib.pyplot as plt

# Pick a random row that requires Treatment (Value 1) to test the hardest cases
test_row = df[df['Treatment'] == 1].iloc[0] 
sample_path = os.path.join(IMAGE_DIR, test_row['image_idx'])

# Process the image through your pipeline
rgb_seq, hsv_seq = prepare_image(sample_path)

# Reconstruct the 1D data (3, 16384) back into a 2D image (128, 128, 3)
reconstructed_rgb = rgb_seq.T.reshape(128, 128, 3)

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB))
plt.title("Original Image (From IMAGE_DIR)")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(reconstructed_rgb)
plt.title("What the AI Sees (Processed Input)")
plt.axis('off')

plt.tight_layout()
plt.show()

print(f"Target Bilirubin: {test_row['blood(mg/dL)']} | Target Treatment: {test_row['Treatment']}")

NameError: name 'prepare_image' is not defined

In [ ]:
# history = model.fit(
#     x={'skin_sequence_input': X_i_tr, 'clinical_input': X_c_tr},
#     y=y_tr,
#     validation_data=({'skin_sequence_input': X_i_ts, 'clinical_input': X_c_ts}, y_ts),
#     epochs=100,
#     batch_size=16
# )

Starting re-tuned training with synchronized dictionaries...
Epoch 1/100


ValueError: Attr 'Toutput_types' of 'OptionalFromValue' Op passed list of length 0 less than minimum 1.

In [ ]:
# # ── 6. EVALUATION ──
# results = model.evaluate(
#     {'rgb_input': X_rgb_test, 'hsv_input': X_hsv_test},
#     {'regression_output': y_reg_test, 'treatment_output': y_tr_test}
# )
# print(f"Final Classification Accuracy: {results[4]*100:.2f}%")

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 23.9164 - regression_output_loss: 23.4687 - regression_output_mae: 3.7598 - treatment_output_accuracy: 0.7673 - treatment_output_loss: 0.4479 
Final Classification Accuracy: 76.73%
